# NG-RF: Neural-Guided Random Forest

Notebook version of `ngrf_experiments.py` + `ngrf_report.py` (identical code).

**Kaggle:** *Add Input* → the three datasets (CIC-IDS2017 `*.pcap_ISCX.csv`, UNSW-NB15 partitioned training/testing CSVs, CSE-CIC-IDS2018 day CSVs). If the file names differ from the default globs in CONFIG, set them in the next cell. Run one dataset per session if needed (`NGRF_DATASETS`); results are checkpointed and resume automatically.

The defaults reproduce the paper (5 repeats, 3 ablation repeats, 100k-flow cap, CNN 20 epochs / patience 3, no BiLSTM/CNN-LSTM). The paper's run took about 11 h on a 2-core CPU without GPU.

In [ ]:
import os, json
# os.environ["NGRF_DATASETS"] = "unsw"          # one dataset per session (optional)
# os.environ["NGRF_CFG_JSON"] = json.dumps({"CSE18_GLOB": "/kaggle/input/<your-dataset>/*.csv"})   # path overrides (optional)

In [ ]:
%%writefile ngrf_report.py
"""Summarise NG-RF experiments -> tokens.json (every number reported in the paper) + figures.
Usage (Kaggle or locally):  python ngrf_report.py /kaggle/working/ngrf_out
"""
import os, sys, json, math, pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

DS = ["cic17", "unsw", "cse18"]
DS_NAME = {"cic17": "CIC-IDS2017", "unsw": "UNSW-NB15", "cse18": "CSE-CIC-IDS2018"}
METHODS = ["RF", "ExtraTrees", "XGBoost", "LightGBM", "CNN", "BiLSTM", "CNN-LSTM",
           "WRF-Gini", "WRF-Perm", "WRF-MI", "WRF-SHAP", "CNN-FS+RF", "NG-RF"]
WRF = ["WRF-Gini", "WRF-Perm", "WRF-MI", "WRF-SHAP"]
UNSW_CLASSES = ["Benign", "Analysis", "Backdoor", "DoS", "Exploits", "Fuzzers", "Generic",
                "Reconnaissance", "Shellcode", "Worms"]
PCT = ["acc", "prec", "rec", "f1", "fpr"]
UNIT = ["auc", "prauc", "mcc"]

def corrected_t(a, b, n_tr, n_te):
    d = np.asarray(a, float) - np.asarray(b, float); J = len(d)
    if J < 2: return float("nan")
    v = np.var(d, ddof=1)
    if v == 0: return 1.0 if d.mean() == 0 else 0.0
    t = d.mean() / math.sqrt((1 / J + n_te / n_tr) * v)
    return float(2 * stats.t.sf(abs(t), J - 1))

def holm(p):
    keys = sorted(p, key=lambda k: p[k]); m = len(keys); out, run = {}, 0.0
    for i, k in enumerate(keys):
        run = max(run, min(1.0, (m - i) * p[k])); out[k] = run
    return out

def ms(v, dec=2):
    v = np.asarray(v, float)
    return f"{v.mean():.{dec}f} ± {v.std(ddof=1) if len(v) > 1 else 0:.{dec}f}"

def fp(p):
    if p is None or (isinstance(p, float) and math.isnan(p)): return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

def phrase(delta, p, higher_is_better=True):
    if p is None or math.isnan(p) or p >= 0.05: return "not significantly different from"
    better = (delta > 0) == higher_is_better
    return "significantly higher than" if delta > 0 else "significantly lower than"

def main(out_dir):
    RAW = pickle.load(open(os.path.join(out_dir, "results_raw.pkl"), "rb"))
    T = {}
    env = RAW.get("env", {})
    T["ENV.cpu"] = str(env.get("cpu", "n/a")); T["ENV.cores"] = str(env.get("cores", "n/a"))
    T["ENV.gpu"] = str(env.get("gpu", "n/a")); T["ENV.ram"] = str(env.get("ram_gb", "n/a"))
    for k in ("python", "torch", "sklearn", "xgboost", "lightgbm"): T[f"ENV.{k}"] = str(env.get(k, "n/a"))
    cfg = env.get("cfg", {})
    T["CFG.R"] = str(cfg.get("REPEATS", "n/a")); T["CFG.RA"] = str(cfg.get("ABL_REPEATS", "n/a"))
    T["CFG.MAXROWS"] = f"{cfg.get('MAX_ROWS', 0):,}"
    T["CFG.EPOCHS"] = str(cfg.get("EPOCHS", "n/a")); T["CFG.PATIENCE"] = str(cfg.get("PATIENCE", "n/a"))
    summary = {}
    for ds in DS:
        if ds not in RAW or "bin" not in RAW[ds]: continue
        info, B, M = RAW[ds]["info"], RAW[ds]["bin"], RAW[ds]["multi"]
        tun = RAW[ds]["tuning"]
        # ---------- dataset table
        T[f"DS.{ds}.n_raw"] = f"{info['n_raw']:,}"; T[f"DS.{ds}.n_dedup"] = f"{info['n_dedup']:,}"
        T[f"DS.{ds}.n_used"] = f"{info['n_used']:,}"; T[f"DS.{ds}.d"] = str(info["d"])
        T[f"DS.{ds}.n_cls"] = str(len(info["class_counts"])); T[f"DS.{ds}.attack"] = f"{100 * info['attack_frac']:.1f}"
        T[f"DS.{ds}.dup_pct"] = f"{100 * (1 - info['n_dedup'] / max(info['n_clean'], 1)):.1f}"
        T[f"DS.{ds}.n_tr"] = f"{B[0]['n_tr']:,}"; T[f"DS.{ds}.n_va"] = f"{B[0]['n_va']:,}"; T[f"DS.{ds}.n_te"] = f"{B[0]['n_te']:,}"
        # ---------- tuned hyper-parameters
        T[f"HP.{ds}.tau"] = f"{tun['tau']:g}"; T[f"HP.{ds}.alpha"] = f"{tun['alpha']:g}"
        T[f"HP.{ds}.m"] = str(tun["m"]); T[f"HP.{ds}.msl"] = str(tun["msl"])
        T[f"HP.{ds}.xgb"] = str(tun["xgb_depth"]); T[f"HP.{ds}.lgbm"] = str(tun["lgbm_leaves"])
        T[f"HP.{ds}.topk"] = f"{100 * tun['topk_frac']:.0f}%"
        for w in WRF: T[f"HP.{ds}.tau.{w}"] = f"{tun['tau_w'][w]:g}"
        # ---------- binary results
        n_tr, n_te = B[0]["n_tr"], B[0]["n_te"]
        pv = {}
        for mth in METHODS:
            if mth not in B[0]["metrics"]:
                for k in PCT + UNIT + ["p"]: T[f"B.{ds}.{mth}.{k}"] = "n/a"
                continue
            for k in PCT: T[f"B.{ds}.{mth}.{k}"] = ms([r["metrics"][mth][k] for r in B])
            for k in UNIT: T[f"B.{ds}.{mth}.{k}"] = ms([r["metrics"][mth][k] for r in B], 3)
            if mth != "NG-RF":
                pv[mth] = corrected_t([r["metrics"]["NG-RF"]["f1"] for r in B], [r["metrics"][mth]["f1"] for r in B], n_tr, n_te)
        ph = holm(pv)
        for mth in METHODS:
            if mth in ph: T[f"B.{ds}.{mth}.p"] = fp(ph[mth])
        T[f"B.{ds}.NG-RF.p"] = "—"
        mean = lambda mth, k, S=B: float(np.mean([r["metrics"][mth][k] for r in S]))
        for mth in ph:
            dlt = mean("NG-RF", "f1") - mean(mth, "f1")
            T[f"CMP.{ds}.{mth}"] = phrase(dlt, ph[mth]); T[f"DF1.{ds}.{mth}"] = f"{dlt:+.2f}"
        best_wrf = max(WRF, key=lambda w: mean(w, "f1")) if all(w in B[0]["metrics"] for w in WRF) else "WRF-Gini"
        best_all = max([m for m in ["RF", "ExtraTrees", "XGBoost", "LightGBM", "CNN", "BiLSTM", "CNN-LSTM", "CNN-FS+RF"] if m in B[0]["metrics"]], key=lambda w: mean(w, "f1"))
        T[f"BESTWRF.{ds}"] = best_wrf; T[f"BESTBASE.{ds}"] = best_all
        T[f"CMP.{ds}.BESTWRF"] = T[f"CMP.{ds}.{best_wrf}"]; T[f"DF1.{ds}.BESTWRF"] = T[f"DF1.{ds}.{best_wrf}"]
        T[f"P.{ds}.BESTWRF"] = T[f"B.{ds}.{best_wrf}.p"]
        T[f"CMP.{ds}.BESTBASE"] = T[f"CMP.{ds}.{best_all}"]; T[f"DF1.{ds}.BESTBASE"] = T[f"DF1.{ds}.{best_all}"]
        T[f"P.{ds}.BESTBASE"] = T[f"B.{ds}.{best_all}.p"]; T[f"P.{ds}.RF"] = T[f"B.{ds}.RF.p"]
        T[f"NG.{ds}.f1"] = f"{mean('NG-RF', 'f1'):.2f}"; T[f"RF.{ds}.f1"] = f"{mean('RF', 'f1'):.2f}"
        T[f"NG.{ds}.fpr"] = f"{mean('NG-RF', 'fpr'):.2f}"; T[f"RF.{ds}.fpr"] = f"{mean('RF', 'fpr'):.2f}"
        rf_fpr, ng_fpr = mean("RF", "fpr"), mean("NG-RF", "fpr")
        T[f"FPRCHG.{ds}"] = f"{100 * (ng_fpr - rf_fpr) / max(rf_fpr, 1e-9):+.1f}%"
        # ---------- efficiency
        for mth in ["RF", "XGBoost", "LightGBM", "CNN", "BiLSTM", "CNN-LSTM", "NG-RF", "WRF-Gini"]:
            if mth not in B[0]["metrics"]:
                for k in ("tr", "te", "sz"): T[f"E.{ds}.{mth}.{k}"] = "n/a"
                continue
            T[f"E.{ds}.{mth}.tr"] = ms([r["metrics"][mth]["train_min"] for r in B])
            T[f"E.{ds}.{mth}.te"] = ms([r["metrics"][mth]["test_ms"] for r in B], 4)
            key = "size_" + mth
            T[f"E.{ds}.{mth}.sz"] = ms([r["extra"][key] for r in B], 1) if key in B[0]["extra"] else "n/a"
        T[f"NG.{ds}.te"] = f"{mean('NG-RF', 'test_ms'):.4f}"; T[f"RF.{ds}.te"] = f"{mean('RF', 'test_ms'):.4f}"
        T[f"NG.{ds}.thr"] = f"{1000 / max(mean('NG-RF', 'test_ms'), 1e-9):,.0f}"
        T[f"CNN.{ds}.params"] = f"{B[0]['extra']['cnn_params']:,}"
        T[f"CNN.{ds}.epochs"] = ms([r["extra"]["cnn_epochs"] for r in B], 1)
        T[f"CNN.{ds}.min"] = ms([r["extra"]["cnn_min"] for r in B])
        T[f"RHO.{ds}"] = ms([r["extra"]["perm_rho"] for r in B], 3)
        T[f"RHOM.{ds}"] = f"{np.mean([r['extra']['perm_rho'] for r in B]):.2f}"
        T[f"TRATIO.{ds}"] = f"{mean('NG-RF', 'train_min') / max(mean('RF', 'train_min'), 1e-9):.0f}"
        T[f"DIV.{ds}.RF"] = ms([r["extra"]["div_RF"] for r in B], 3); T[f"DIV.{ds}.NG-RF"] = ms([r["extra"]["div_NG-RF"] for r in B], 3)
        # ---------- multi-class
        n_tr, n_te = M[0]["n_tr"], M[0]["n_te"]; pv = {}
        for mth in METHODS:
            if mth not in M[0]["metrics"]: T[f"M.{ds}.{mth}.mf1"] = T[f"M.{ds}.{mth}.acc"] = T[f"M.{ds}.{mth}.p"] = "n/a"; continue
            T[f"M.{ds}.{mth}.mf1"] = ms([r["metrics"][mth]["macro_f1"] for r in M])
            T[f"M.{ds}.{mth}.acc"] = ms([r["metrics"][mth]["acc"] for r in M])
            if mth != "NG-RF":
                pv[mth] = corrected_t([r["metrics"]["NG-RF"]["macro_f1"] for r in M], [r["metrics"][mth]["macro_f1"] for r in M], n_tr, n_te)
        for k, v in holm(pv).items(): T[f"M.{ds}.{k}.p"] = fp(v)
        T[f"M.{ds}.NG-RF.p"] = "—"
        mm = lambda mth: float(np.mean([r["metrics"][mth]["macro_f1"] for r in M]))
        T[f"MNG.{ds}"] = f"{mm('NG-RF'):.2f}"; T[f"MRF.{ds}"] = f"{mm('RF'):.2f}"
        T[f"DMF1.{ds}"] = f"{mm('NG-RF') - mm('RF'):.1f}"
        T[f"MCMP.{ds}.RF"] = phrase(mm("NG-RF") - mm("RF"), holm(pv).get("RF", float("nan")))
        if ds == "unsw":
            gains = []
            for c in UNSW_CLASSES:
                for mth, tag in (("RF", "RF"), ("NG-RF", "NG")):
                    for k in ("p", "r", "f1"):
                        vals = [r["metrics"][mth]["per_class"][c][k] for r in M if c in r["metrics"][mth]["per_class"]]
                        T[f"PC.{c}.{tag}.{k}"] = f"{np.mean(vals):.2f}" if vals else "n/a"
                n = [r["metrics"]["RF"]["per_class"][c]["n"] for r in M if c in r["metrics"]["RF"]["per_class"]]
                T[f"PC.{c}.n"] = f"{int(np.mean(n)):,}" if n else "n/a"
                if T[f"PC.{c}.NG.f1"] != "n/a":
                    gains.append((float(T[f"PC.{c}.NG.f1"]) - float(T[f"PC.{c}.RF.f1"]), c))
            gains.sort(reverse=True)
            for i, (g, c) in enumerate(gains[:3]): T[f"PCG{i + 1}.name"] = c; T[f"PCG{i + 1}.val"] = f"{g:+.2f}"
            T["PCW.name"] = gains[-1][1]; T["PCW.val"] = f"{gains[-1][0]:+.2f}"
            T["PC.n_improved"] = str(sum(g > 0 for g, _ in gains)); T["PC.n_total"] = str(len(gains))
            T["PC.macro.RF"] = T["MRF.unsw"]; T["PC.macro.NG"] = T["MNG.unsw"]
        # ---------- ablations
        if "abl" in RAW[ds]:
            A = RAW[ds]["abl"]
            for a in A[0]["alpha"]:
                T[f"A.{ds}.alpha.{a:g}.f1"] = ms([x["alpha"][a]["f1"] for x in A]); T[f"A.{ds}.alpha.{a:g}.fpr"] = ms([x["alpha"][a]["fpr"] for x in A])
            for t in A[0]["tau"]:
                tk = "inf" if math.isinf(t) else f"{t:g}"
                T[f"A.{ds}.tau.{tk}.f1"] = ms([x["tau"][t]["f1"] for x in A]); T[f"A.{ds}.tau.{tk}.div"] = ms([x["tau"][t]["div"] for x in A], 3)
            for k in A[0]["K"]:
                T[f"A.{ds}.K.{k}.f1"] = ms([x["K"][k]["f1"] for x in A]); T[f"A.{ds}.K.{k}.tr"] = ms([x["K"][k]["train_min"] for x in A])
                T[f"A.{ds}.K.{k}.sz"] = ms([x["K"][k]["size"] for x in A], 1)
            for i, nm in enumerate(A[0]["depth"]):
                T[f"A.{ds}.depth.{i}.f1"] = ms([x["depth"][nm]["f1"] for x in A]); T[f"A.{ds}.depth.{i}.params"] = f"{A[0]['depth'][nm]['params']:,}"
                T[f"A.{ds}.depth.{i}.min"] = ms([x["depth"][nm]["cnn_min"] for x in A])
            T[f"A.{ds}.perm.1.f1"] = ms([x["perm"][1]["f1"] for x in A])
            kp = [k for k in A[0]["perm"] if k != 1][0]
            T[f"A.{ds}.perm.P.f1"] = ms([x["perm"][kp]["f1"] for x in A if x["perm"][kp]]) if A[0]["perm"][kp] else "n/a"
            T[f"A.{ds}.rho"] = ms([x["perm_rho"] for x in A], 3)
        summary[ds] = dict(B=B, M=M, info=info)

    json.dump(T, open(os.path.join(out_dir, "tokens.json"), "w"), indent=1, ensure_ascii=False)
    make_figures(RAW, out_dir)
    # summary of the main comparison
    print("\n=== SUMMARY (binary F1, corrected resampled t-test with Holm adjustment) ===")
    for ds in DS:
        if f"NG.{ds}.f1" in T:
            print(f"{DS_NAME[ds]}: NG-RF {T[f'NG.{ds}.f1']} | RF {T[f'RF.{ds}.f1']} (p={T[f'P.{ds}.RF']}) | "
                  f"best WRF {T[f'BESTWRF.{ds}']} ΔF1 {T[f'DF1.{ds}.BESTWRF']} p={T[f'P.{ds}.BESTWRF']} | "
                  f"best baseline {T[f'BESTBASE.{ds}']} ΔF1 {T[f'DF1.{ds}.BESTBASE']} p={T[f'P.{ds}.BESTBASE']}")
    print("tokens:", len(T), "->", os.path.join(out_dir, "tokens.json"))

# ------------------------------------------------------------------ FIGURES
C = {"RF": "#9aa5b1", "XGBoost": "#5b8db8", "LightGBM": "#7fb3a6", "WRF-Gini": "#d9a441", "WRF-SHAP": "#c77c48", "NG-RF": "#b03a48"}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

def make_figures(RAW, out):
    ds_ok = [d for d in DS if d in RAW and "bin" in RAW[d]]
    # Fig 2: F1 and FPR per dataset
    fig, ax = plt.subplots(1, 2, figsize=(6.5, 2.8), dpi=300)
    meths = [m for m in C if m in RAW[ds_ok[0]]["bin"][0]["metrics"]]
    w = 0.8 / len(meths); x = np.arange(len(ds_ok))
    for j, (k, lab) in enumerate((("f1", "F1-score (%)"), ("fpr", "False positive rate (%)"))):
        for i, m in enumerate(meths):
            v = [[r["metrics"][m][k] for r in RAW[d]["bin"]] for d in ds_ok]
            ax[j].bar(x + (i - len(meths) / 2 + 0.5) * w, [np.mean(a) for a in v], w, yerr=[np.std(a, ddof=1) if len(a) > 1 else 0 for a in v],
                      color=C[m], label=m, capsize=1.5, error_kw=dict(lw=0.6))
        ax[j].set_xticks(x); ax[j].set_xticklabels([DS_NAME[d] for d in ds_ok], fontsize=7.5); ax[j].set_ylabel(lab)
        if k == "f1":
            lo = min(np.mean([r["metrics"][m]["f1"] for r in RAW[d]["bin"]]) for d in ds_ok for m in meths)
            ax[j].set_ylim(max(0, lo - 3), 100)
    h, l = ax[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=len(meths), fontsize=6.5, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.92)); fig.savefig(os.path.join(out, "fig2_binary.png")); plt.close(fig)
    # Fig 3: per-class F1 (UNSW-NB15, multi-class)
    fig, ax = plt.subplots(figsize=(6.5, 2.8), dpi=300)
    if "unsw" in RAW and "multi" in RAW["unsw"]:
        M = RAW["unsw"]["multi"]; cls = [c for c in UNSW_CLASSES if c in M[0]["metrics"]["RF"]["per_class"]]
        x = np.arange(len(cls))
        for i, (m, col) in enumerate((("RF", C["RF"]), ("NG-RF", C["NG-RF"]))):
            v = [[r["metrics"][m]["per_class"][c]["f1"] for r in M] for c in cls]
            ax.bar(x + (i - 0.5) * 0.38, [np.mean(a) for a in v], 0.38, yerr=[np.std(a, ddof=1) if len(a) > 1 else 0 for a in v],
                   color=col, label=m, capsize=1.5, error_kw=dict(lw=0.6))
        ax.set_xticks(x); ax.set_xticklabels(cls, rotation=30, ha="right", fontsize=7.5); ax.set_ylabel("Per-class F1 (%)")
        ax.legend(frameon=False, fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(out, "fig3_perclass.png")); plt.close(fig)
    # Fig 4: sampling probability vs Gini importance, top-15 features (first dataset)
    d0 = ds_ok[0]; B = RAW[d0]["bin"]; feats = B[0]["features"]
    P = np.mean([r["extra"]["P"] for r in B], 0); Gi = np.mean([r["extra"]["W"]["WRF-Gini"] for r in B], 0)
    U_rf = np.mean([r["extra"]["usage_RF"] for r in B], 0); U_ng = np.mean([r["extra"]["usage_NG-RF"] for r in B], 0)
    top = np.argsort(-P)[:15][::-1]
    fig, ax = plt.subplots(1, 2, figsize=(6.5, 3.4), dpi=300, sharey=True)
    y = np.arange(len(top))
    ax[0].barh(y - 0.2, P[top] / P.max(), 0.4, color=C["NG-RF"], label="NG-RF sampling prob. $P_\\tau$")
    ax[0].barh(y + 0.2, Gi[top] / Gi.max(), 0.4, color=C["WRF-Gini"], label="RF Gini importance")
    ax[0].set_yticks(y); ax[0].set_yticklabels([feats[i][:28] for i in top], fontsize=6.5); ax[0].set_xlabel("Normalised score")
    ax[0].legend(fontsize=6, frameon=False, loc="lower right")
    ax[1].barh(y - 0.2, U_ng[top] * 100, 0.4, color=C["NG-RF"], label="NG-RF")
    ax[1].barh(y + 0.2, U_rf[top] * 100, 0.4, color=C["RF"], label="RF")
    ax[1].set_xlabel("Share of split nodes (%)"); ax[1].legend(fontsize=6, frameon=False, loc="lower right")
    fig.suptitle(DS_NAME[d0], fontsize=8); fig.tight_layout(); fig.savefig(os.path.join(out, "fig4_importance.png")); plt.close(fig)
    # Fig 5: tau sensitivity
    fig, ax = plt.subplots(1, 2, figsize=(6.5, 2.6), dpi=300)
    for d, mk in zip(ds_ok, "osd"):
        if "abl" not in RAW[d]: continue
        A = RAW[d]["abl"]; ts = list(A[0]["tau"]); xs = np.arange(len(ts))
        ax[0].errorbar(xs, [np.mean([a["tau"][t]["f1"] for a in A]) for t in ts], [np.std([a["tau"][t]["f1"] for a in A]) for t in ts], marker=mk, ms=3, lw=1, capsize=2, label=DS_NAME[d])
        ax[1].plot(xs, [np.mean([a["tau"][t]["div"] for a in A]) for t in ts], marker=mk, ms=3, lw=1, label=DS_NAME[d])
        lab = ["∞" if math.isinf(t) else f"{t:g}" for t in ts]
    for a_, yl in zip(ax, ("F1-score (%)", "Tree disagreement")):
        a_.set_xticks(xs); a_.set_xticklabels(lab); a_.set_xlabel("Temperature τ"); a_.set_ylabel(yl)
    ax[0].legend(fontsize=6.5, frameon=False)
    fig.tight_layout(); fig.savefig(os.path.join(out, "fig5_tau.png")); plt.close(fig)

if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else "/kaggle/working/ngrf_out")

# NG-RF: Neural-Guided Random Forest - full experimental pipeline

Produces every result, table value and figure reported in the paper.

* Datasets: CIC-IDS2017, UNSW-NB15 (partitioned sets), CSE-CIC-IDS2018
* Protocol: 5 repeated stratified 60/20/20 train/validation/test splits.
  All hyper-parameters (tau, alpha, RF/XGB/LGBM settings) are chosen on the
  validation partition of repeat 0 and then frozen. Test partitions are used once.
* Output: OUT_DIR/{results_raw.pkl, tokens.json, fig*.png}; tokens.json holds every number in the paper.

Every (dataset, repeat) is checkpointed in OUT_DIR/ckpt, so an interrupted run resumes where it stopped.
To split the work, run one dataset per session (NGRF_DATASETS=unsw, ...) and merge with merge_results.py.

In [ ]:
# %%
import os, sys, gc, json, time, glob, math, pickle, platform, warnings, itertools, subprocess
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, matthews_corrcoef,
                             confusion_matrix, precision_recall_fscore_support)
from scipy import stats

import torch
import torch.nn as nn

In [ ]:
# %% ---------------------------------------------------------------- CONFIG
CFG = dict(
    # paths (recursive globs). Override with NGRF_CFG_JSON, e.g. '{"CIC17_GLOB": "data/cic17/*.csv"}'.
    # Defaults below are the settings used for every result in the paper.
    CIC17_GLOB = "/kaggle/input/**/*.pcap_ISCX.csv",
    UNSW_TRAIN = "/kaggle/input/**/UNSW_NB15_training-set.csv",
    UNSW_TEST  = "/kaggle/input/**/UNSW_NB15_testing-set.csv",
    CSE18_GLOB = "/kaggle/input/**/*TrafficForML_CICFlowMeter.csv",
    OUT_DIR    = "/kaggle/working/ngrf_out",

    DATASETS   = os.environ.get("NGRF_DATASETS", "cic17,unsw,cse18").split(","),
    REPEATS    = int(os.environ.get("NGRF_REPEATS", 5)),
    ABL_REPEATS= int(os.environ.get("NGRF_ABL_REPEATS", 3)),
    MAX_ROWS   = int(os.environ.get("NGRF_MAX_ROWS", 100_000)),   # after cleaning/dedup, stratified cap
    CSE18_FRAC_BENIGN = 0.05, CSE18_FRAC_ATTACK = 0.35,            # chunk sampling while reading 16M rows
    MIN_CLASS  = 50,                                                # smaller classes merged into "Other"

    K          = 100,           # trees in all forests (main results)
    POOL_MULT  = 3,             # replication pool size = POOL_MULT * d
    TAU_GRID   = [0.5, 1.0, 2.0, 5.0, 10.0],
    ALPHA_GRID = [0.0, 0.25, 0.5, 0.75, 1.0],
    TUNE_ROWS  = 150_000,       # train rows used during validation tuning
    TUNE_K     = 50,

    CNN_CHANNELS = (64, 128, 256), CNN_FC = (512, 128), DROPOUT = 0.3,
    EPOCHS = 20, PATIENCE = 3, BATCH = 256, LR = 1e-3, WD = 1e-4,
    N_PERM = 3,                 # CNNs trained on random column orders (order invariance)
    N_IMP  = 20_000,            # training samples used to compute importances
    RUN_SEQ_BASELINES = False,  # BiLSTM & CNN-LSTM (not used in the paper; needs a GPU to be fast)
    RUN_ABLATIONS = True,
    SEED = 42,
)
CFG.update(json.loads(os.environ.get("NGRF_CFG_JSON", "{}")))   # optional overrides
os.makedirs(CFG["OUT_DIR"], exist_ok=True)
CKPT = os.path.join(CFG["OUT_DIR"], "ckpt"); os.makedirs(CKPT, exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_JOBS = -1
print("device:", DEV, "| cpu:", platform.processor() or platform.machine(), "| cores:", os.cpu_count())

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

def seed_all(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

In [ ]:
# %% ---------------------------------------------------------------- DATA LOADING
def _glob(p):
    f = sorted(glob.glob(p, recursive=True))
    if not f: raise FileNotFoundError(p)
    return f

DROP_ID_COLS = {"flow id", "src ip", "source ip", "dst ip", "destination ip", "src port", "source port",
                "dst port", "destination port", "timestamp", "protocol_name"}

CIC17_MAP = {"BENIGN": "Benign", "DoS Hulk": "DoS", "DoS GoldenEye": "DoS", "DoS slowloris": "DoS",
             "DoS Slowhttptest": "DoS", "DDoS": "DDoS", "PortScan": "PortScan", "FTP-Patator": "BruteForce",
             "SSH-Patator": "BruteForce", "Bot": "Bot", "Infiltration": "Infiltration", "Heartbleed": "Heartbleed"}
def map_cic17(l):
    l = str(l).strip()
    if l.startswith("Web Attack"): return "WebAttack"
    return CIC17_MAP.get(l, l)

def map_cse18(l):
    l = str(l).strip()
    if l == "Benign": return "Benign"
    if l.startswith("DDOS") or l.startswith("DDoS"): return "DDoS"
    if l.startswith("DoS"): return "DoS"
    if l in ("FTP-BruteForce", "SSH-Bruteforce"): return "BruteForce"
    if l.startswith("Brute Force") or l == "SQL Injection": return "WebAttack"
    if l == "Bot": return "Bot"
    if l == "Infilteration": return "Infiltration"
    return l

def clean_numeric(df, label_col):
    df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
    y = df[label_col].astype(str)
    X = df.drop(columns=[c for c in df.columns if c == label_col or c.lower() in DROP_ID_COLS])
    X = X.apply(pd.to_numeric, errors="coerce").astype("float64")
    X = X.replace([np.inf, -np.inf], np.nan)
    ok = X.notna().all(axis=1)
    return X[ok].astype("float32"), y[ok]

def load_cic17():
    Xs, ys, n_raw = [], [], 0
    for f in _glob(CFG["CIC17_GLOB"]):                      # clean file by file to bound memory
        d = pd.read_csv(f, low_memory=False, encoding="latin-1")
        n_raw += len(d)
        X, y = clean_numeric(d, "Label"); del d
        Xs.append(X); ys.append(y.map(map_cic17))
    cols = Xs[0].columns
    X = pd.concat([x[cols] for x in Xs], ignore_index=True); y = pd.concat(ys, ignore_index=True)
    return X, y, None, n_raw

def load_unsw():
    df = pd.concat([pd.read_csv(_glob(CFG["UNSW_TRAIN"])[0]), pd.read_csv(_glob(CFG["UNSW_TEST"])[0])], ignore_index=True)
    n_raw = len(df)
    df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
    df = df.drop(columns=[c for c in ["id", "label"] if c in df.columns])
    df["attack_cat"] = df["attack_cat"].fillna("Normal").astype(str).str.strip().replace({"Normal": "Benign", "Backdoors": "Backdoor"})
    cats = ["proto", "service", "state"]
    y = df.pop("attack_cat")
    for c in df.columns:
        if c not in cats: df[c] = pd.to_numeric(df[c], errors="coerce")
    ok = df.drop(columns=cats).notna().all(axis=1)
    return df[ok], y[ok], cats, n_raw

def load_cse18():
    parts, n_raw = [], 0
    rng = np.random.default_rng(CFG["SEED"])
    for f in _glob(CFG["CSE18_GLOB"]):
        for ch in pd.read_csv(f, chunksize=500_000, low_memory=False):
            ch.columns = [c.strip() for c in ch.columns]
            ch = ch[ch["Label"].astype(str) != "Label"]          # repeated header rows
            n_raw += len(ch)
            ben = ch["Label"].astype(str).str.strip() == "Benign"
            frac = np.where(ben, CFG["CSE18_FRAC_BENIGN"], CFG["CSE18_FRAC_ATTACK"])
            keep = rng.random(len(ch)) < frac
            ch = ch[keep]
            X, y = clean_numeric(ch, "Label")
            X["__y"] = y.map(map_cse18).values
            parts.append(X)
        log("read", os.path.basename(f), "rows so far", n_raw)
    df = pd.concat(parts, ignore_index=True); del parts
    common = [c for c in df.columns if c != "__y"]
    df = df.dropna(subset=common)                                 # files with extra columns
    y = df.pop("__y")
    return df, y, None, n_raw

LOADERS = {"cic17": load_cic17, "unsw": load_unsw, "cse18": load_cse18}
DS_NAME = {"cic17": "CIC-IDS2017", "unsw": "UNSW-NB15", "cse18": "CSE-CIC-IDS2018"}

def prepare(ds):
    cache = os.path.join(CKPT, f"{ds}_data.pkl")
    if os.path.exists(cache):
        return pickle.load(open(cache, "rb"))
    log("loading", ds)
    X, y, cats, n_raw = LOADERS[ds]()
    X = X.reset_index(drop=True); y = y.reset_index(drop=True)
    n_clean = len(X)
    # duplicate removal (identical feature vector + label)
    dup = pd.concat([X, y.rename("__y")], axis=1).duplicated()
    X, y = X[~dup].reset_index(drop=True), y[~dup].reset_index(drop=True)
    n_dedup = len(X)
    # conflicting duplicates (same features, different labels) are removed as label noise
    conf = X.duplicated(keep=False)
    X, y = X[~conf].reset_index(drop=True), y[~conf].reset_index(drop=True)
    # drop constant columns
    num = [c for c in X.columns if cats is None or c not in cats]
    const = [c for c in num if X[c].nunique() <= 1]
    X = X.drop(columns=const)
    # merge tiny classes
    vc = y.value_counts()
    y = y.where(~y.isin(vc[vc < CFG["MIN_CLASS"]].index), "Other")
    # stratified cap
    n = len(X)
    if n > CFG["MAX_ROWS"]:
        frac = CFG["MAX_ROWS"] / n
        rng = np.random.default_rng(CFG["SEED"]); idx = []
        for c, g in y.groupby(y).groups.items():
            g = np.asarray(g); k = max(int(round(frac * len(g))), min(len(g), 5000))
            idx.append(rng.choice(g, size=min(k, len(g)), replace=False))
        idx = np.sort(np.concatenate(idx))
        X, y = X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)
    info = dict(n_raw=int(n_raw), n_clean=int(n_clean), n_dedup=int(n_dedup), n_conflict=int(conf.sum()),
                n_const=len(const), n_used=len(X), d=X.shape[1],
                class_counts={k: int(v) for k, v in y.value_counts().items()},
                attack_frac=float((y != "Benign").mean()))
    log(ds, info)
    out = (X, y, cats, info)
    pickle.dump(out, open(cache, "wb"), protocol=4)
    return out

def make_split(X, y, rep):
    s = CFG["SEED"] + rep
    idx = np.arange(len(y))
    tr, te = train_test_split(idx, test_size=0.2, stratify=y, random_state=s)
    tr, va = train_test_split(tr, test_size=0.25, stratify=y.iloc[tr], random_state=s)
    return tr, va, te

class Prep:
    """Frequency-encode categoricals, signed-log transform, standardise. Fitted on TRAIN only."""
    def __init__(self, cats): self.cats = cats or []
    def fit(self, X):
        self.freq = {c: X[c].astype(str).value_counts(normalize=True) for c in self.cats}
        Z = self._t(X); self.sc = StandardScaler().fit(Z); self.cols = list(X.columns); return self
    def _t(self, X):
        X = X.copy()
        for c in self.cats: X[c] = X[c].astype(str).map(self.freq[c]).fillna(0.0)
        Z = X.values.astype("float64")
        return np.sign(Z) * np.log1p(np.abs(Z))
    def transform(self, X):
        return np.clip(self.sc.transform(self._t(X)), -10, 10).astype("float32")

In [ ]:
# %% ---------------------------------------------------------------- METRICS
def binary_metrics(y, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(acc=100 * accuracy_score(y, pred), prec=100 * precision_score(y, pred, zero_division=0),
                rec=100 * recall_score(y, pred), f1=100 * f1_score(y, pred), fpr=100 * fp / max(fp + tn, 1),
                auc=roc_auc_score(y, prob), prauc=average_precision_score(y, prob), mcc=matthews_corrcoef(y, pred))

def multiclass_metrics(y, pred, classes):
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=np.arange(len(classes)), zero_division=0)
    return dict(acc=100 * accuracy_score(y, pred), macro_f1=100 * f.mean(),
                per_class={c: dict(p=100 * p[i], r=100 * r[i], f1=100 * f[i], n=int(s[i])) for i, c in enumerate(classes)})

In [ ]:
# %% ---------------------------------------------------------------- CNN FEATURE LEARNER
class CNN1D(nn.Module):
    def __init__(self, d, n_cls, chans=(64, 128, 256), fc=(512, 128), p=0.3):
        super().__init__()
        blocks, cin, L = [], 1, d
        for c in chans:
            blocks.append(nn.ModuleDict(dict(conv=nn.Conv1d(cin, c, 3, padding=1), bn=nn.BatchNorm1d(c))))
            cin, L = c, math.ceil(L / 2)
        self.blocks = nn.ModuleList(blocks)
        self.pool = nn.MaxPool1d(2, ceil_mode=True)
        layers, fin = [nn.Flatten(), nn.Dropout(p)], cin * L
        for h in fc: layers += [nn.Linear(fin, h), nn.ReLU(), nn.Dropout(p)]; fin = h
        layers.append(nn.Linear(fin, n_cls))
        self.head = nn.Sequential(*layers)
    def forward(self, x, return_act=False):
        h = x.unsqueeze(1); act = None
        for i, b in enumerate(self.blocks):
            h = torch.relu(b["bn"](b["conv"](h)))
            if i == 0: act = h                      # first-block responses, shape (n, c1, d)
            h = self.pool(h)
        out = self.head(h)
        return (out, act) if return_act else out

def n_params(m): return int(sum(p.numel() for p in m.parameters()))

def class_weights(y, n_cls):
    cnt = np.bincount(y, minlength=n_cls).astype("float64")
    w = 1.0 / np.sqrt(np.maximum(cnt, 1)); w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=DEV)

def train_net(net, Xtr, ytr, Xva, yva, n_cls, seed):
    seed_all(seed); net = net.to(DEV)
    opt = torch.optim.Adam(net.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    lossf = nn.CrossEntropyLoss(weight=class_weights(ytr, n_cls))
    Xt, yt = torch.tensor(Xtr), torch.tensor(ytr, dtype=torch.long)
    Xv, yv = torch.tensor(Xva).to(DEV), torch.tensor(yva, dtype=torch.long).to(DEV)
    best, best_state, bad, ep_run = 1e9, None, 0, 0
    t0 = time.time()
    for ep in range(CFG["EPOCHS"]):
        net.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), CFG["BATCH"]):
            b = perm[i:i + CFG["BATCH"]]
            xb, yb = Xt[b].to(DEV), yt[b].to(DEV)
            opt.zero_grad(); l = lossf(net(xb), yb); l.backward(); opt.step()
        net.eval(); vl = 0.0
        with torch.no_grad():
            for i in range(0, len(Xv), 8192): vl += lossf(net(Xv[i:i + 8192]), yv[i:i + 8192]).item() * len(Xv[i:i + 8192])
        vl /= len(Xv); ep_run = ep + 1
        if vl < best - 1e-5: best, bad, best_state = vl, 0, {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG["PATIENCE"]: break
    net.load_state_dict(best_state); net.eval()
    return net, time.time() - t0, ep_run

def net_proba(net, X):
    out = []
    with torch.no_grad():
        for i in range(0, len(X), 8192):
            out.append(torch.softmax(net(torch.tensor(X[i:i + 8192]).to(DEV)), 1).cpu().numpy())
    return np.concatenate(out)

def cnn_importance(net, X, y, n_cls):
    """Gradient importance (Eq. 16) and activation importance (Eq. 18) on a training subset."""
    lossf = nn.CrossEntropyLoss(weight=class_weights(y, n_cls), reduction="sum")
    g_sum = np.zeros(X.shape[1]); a_sum = np.zeros(X.shape[1])
    for i in range(0, len(X), 4096):
        xb = torch.tensor(X[i:i + 4096]).to(DEV).requires_grad_(True)
        yb = torch.tensor(y[i:i + 4096], dtype=torch.long).to(DEV)
        out, act = net(xb, return_act=True)
        lossf(out, yb).backward()
        g_sum += xb.grad.abs().sum(0).detach().cpu().numpy()
        a_sum += act.detach().norm(dim=1).sum(0).cpu().numpy()
    return g_sum / len(X), a_sum / len(X)

def learn_importances(Xtr, ytr, Xva, yva, n_cls, seed, n_perm=None, chans=None):
    """Train n_perm CNNs on random column orders; return averaged normalised importances."""
    n_perm = n_perm or CFG["N_PERM"]; chans = chans or CFG["CNN_CHANNELS"]
    d = Xtr.shape[1]; rng = np.random.default_rng(seed)
    sub = rng.choice(len(Xtr), size=min(CFG["N_IMP"], len(Xtr)), replace=False)
    G, A, nets, secs = [], [], [], 0.0
    for p in range(n_perm):
        pi = np.arange(d) if p == 0 else rng.permutation(d)
        net = CNN1D(d, n_cls, chans, CFG["CNN_FC"], CFG["DROPOUT"])
        net, t, ep = train_net(net, Xtr[:, pi], ytr, Xva[:, pi], yva, n_cls, seed + p)
        secs += t
        g, a = cnn_importance(net, Xtr[sub][:, pi], ytr[sub], n_cls)
        gi, ai = np.zeros(d), np.zeros(d); gi[pi] = g; ai[pi] = a      # map back to original order
        G.append(gi / gi.max()); A.append(ai / ai.max()); nets.append((net, pi, ep))
    rho = [stats.spearmanr(G[i] * 0.5 + A[i] * 0.5, G[j] * 0.5 + A[j] * 0.5)[0] for i, j in itertools.combinations(range(n_perm), 2)]
    return dict(G=np.mean(G, 0), A=np.mean(A, 0), G_list=G, A_list=A, nets=nets, cnn_secs=secs,
                perm_rho=float(np.mean(rho)) if rho else float("nan"))

def combine(G, A, alpha):
    I = alpha * G / G.max() + (1 - alpha) * A / A.max()
    return np.maximum(I, 1e-3)

def to_prob(I, tau):
    if tau is None or np.isinf(tau): return np.full(len(I), 1.0 / len(I))
    w = np.power(I / I.max(), 1.0 / tau); return w / w.sum()

In [ ]:
# %% ---------------------------------------------------------------- WEIGHTED-SAMPLING FOREST
class WeightedRF:
    """Random forest whose per-node candidate features are drawn with probability P(j).
    Implemented exactly through column replication: feature j appears r_j = max(1, round(M*P(j)))
    times in a pool of M = POOL_MULT*d columns; scikit-learn draws m candidate columns per node
    uniformly from the pool, so feature j is drawn with probability r_j / sum(r)."""
    def __init__(self, P=None, K=100, m=None, min_samples_leaf=1, seed=0, forest="rf"):
        self.P, self.K, self.m, self.msl, self.seed, self.forest = P, K, m, min_samples_leaf, seed, forest
    def _idx(self, d):
        if self.P is None: return np.arange(d)
        M = CFG["POOL_MULT"] * d
        r = np.maximum(1, np.round(self.P * M)).astype(int)
        return np.repeat(np.arange(d), r)
    def fit(self, X, y):
        d = X.shape[1]; self.idx = self._idx(d)
        m = self.m or max(1, int(round(math.sqrt(d))))
        Cls = RandomForestClassifier if self.forest == "rf" else ExtraTreesClassifier
        self.model = Cls(n_estimators=self.K, max_features=min(m, len(self.idx)), min_samples_leaf=self.msl,
                         n_jobs=N_JOBS, random_state=self.seed)
        self.model.fit(X[:, self.idx], y); return self
    def predict_proba(self, X): return self.model.predict_proba(X[:, self.idx])
    def size_mb(self): return len(pickle.dumps(self.model, protocol=4)) / 2 ** 20
    def diversity(self, X, n_pairs=30, n=5000, seed=0):
        rng = np.random.default_rng(seed); Xs = X[rng.choice(len(X), min(n, len(X)), replace=False)][:, self.idx]
        est = self.model.estimators_; preds = {}
        dis = []
        for _ in range(n_pairs):
            i, j = rng.choice(len(est), 2, replace=False)
            for k in (i, j):
                if k not in preds: preds[k] = est[k].predict(Xs)
            dis.append((preds[i] != preds[j]).mean())
        return float(np.mean(dis))
    def feature_usage(self, d):
        cnt = np.zeros(d)
        for t in self.model.estimators_:
            f = t.tree_.feature; f = f[f >= 0]
            np.add.at(cnt, self.idx[f], 1)
        return cnt / cnt.sum()

def fit_eval_forest(P, Xtr, ytr, Xte, yte, K, m, msl, seed, forest="rf", binary=True, classes=None):
    t0 = time.time(); f = WeightedRF(P, K, m, msl, seed, forest).fit(Xtr, ytr); ttr = time.time() - t0
    t0 = time.time(); pr = f.predict_proba(Xte); tte = (time.time() - t0) / len(Xte) * 1e3
    res = binary_metrics(yte, pr[:, 1]) if binary else multiclass_metrics(yte, pr.argmax(1), classes)
    res.update(train_min=ttr / 60, test_ms=tte)
    return f, res

In [ ]:
# %% ---------------------------------------------------------------- OTHER BASELINES
def run_xgb(Xtr, ytr, Xva, yva, Xte, n_cls, depth, seed):
    import xgboost as xgb
    kw = dict(n_estimators=1000, learning_rate=0.1, max_depth=depth, tree_method="hist", random_state=seed,
              n_jobs=N_JOBS, early_stopping_rounds=30, eval_metric="logloss" if n_cls == 2 else "mlogloss")
    if torch.cuda.is_available(): kw["device"] = "cuda"
    m = xgb.XGBClassifier(**kw)
    t0 = time.time(); m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False); ttr = time.time() - t0
    t0 = time.time(); pr = m.predict_proba(Xte); tte = (time.time() - t0) / len(Xte) * 1e3
    return m, pr, ttr, tte

def run_lgbm(Xtr, ytr, Xva, yva, Xte, n_cls, leaves, seed):
    import lightgbm as lgb
    m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=leaves, random_state=seed,
                           n_jobs=N_JOBS, verbose=-1)
    t0 = time.time(); m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)]); ttr = time.time() - t0
    t0 = time.time(); pr = m.predict_proba(Xte); tte = (time.time() - t0) / len(Xte) * 1e3
    return m, pr, ttr, tte

class BiLSTM(nn.Module):
    def __init__(self, d, n_cls, h=64):
        super().__init__(); self.lstm = nn.LSTM(1, h, num_layers=2, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(nn.Linear(2 * h, 64), nn.ReLU(), nn.Linear(64, n_cls))
    def forward(self, x):
        o, _ = self.lstm(x.unsqueeze(-1)); return self.fc(o[:, -1])

class CNNLSTM(nn.Module):
    def __init__(self, d, n_cls, h=64):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2, ceil_mode=True))
        self.lstm = nn.LSTM(64, h, batch_first=True); self.fc = nn.Linear(h, n_cls)
    def forward(self, x):
        h = self.conv(x.unsqueeze(1)).transpose(1, 2); o, _ = self.lstm(h); return self.fc(o[:, -1])

def eval_net(net, Xte, yte, binary, classes):
    t0 = time.time(); pr = net_proba(net, Xte); tte = (time.time() - t0) / len(Xte) * 1e3
    res = binary_metrics(yte, pr[:, 1]) if binary else multiclass_metrics(yte, pr.argmax(1), classes)
    res["test_ms"] = tte; return res

def importance_baselines(rf_uniform, Xtr, ytr, Xva, yva, xgb_model, seed):
    """Non-neural feature weights for the closest-work weighted forests."""
    d = Xtr.shape[1]; rng = np.random.default_rng(seed)
    W = {}
    W["WRF-Gini"] = rf_uniform.model.feature_importances_.copy()
    sub = rng.choice(len(Xva), min(10000, len(Xva)), replace=False)
    pi = permutation_importance(rf_uniform.model, Xva[sub][:, rf_uniform.idx], yva[sub], n_repeats=3, random_state=seed, n_jobs=1,
                                scoring="f1_macro")
    W["WRF-Perm"] = np.maximum(pi.importances_mean, 0)
    sub = rng.choice(len(Xtr), min(50000, len(Xtr)), replace=False)
    W["WRF-MI"] = mutual_info_classif(Xtr[sub], ytr[sub], random_state=seed)
    import xgboost as xgb
    sub = rng.choice(len(Xva), min(5000, len(Xva)), replace=False)
    contrib = xgb_model.get_booster().predict(xgb.DMatrix(Xva[sub]), pred_contribs=True)
    contrib = np.abs(contrib).sum(1) if contrib.ndim == 3 else np.abs(contrib)   # multi-class: (n, C, d+1)
    W["WRF-SHAP"] = contrib[:, :d].mean(0) if contrib.ndim == 2 else contrib[:d]
    for k in W: W[k] = np.maximum(W[k] / max(W[k].max(), 1e-12), 1e-3)
    return W

In [ ]:
# %% ---------------------------------------------------------------- TUNING (repeat 0, validation only)
def tune(ds, Xtr, ytr, Xva, yva, n_cls, imp, W):
    path = os.path.join(CKPT, f"{ds}_tuning.json")
    if os.path.exists(path): return json.load(open(path))
    rng = np.random.default_rng(0)
    s = rng.choice(len(Xtr), min(CFG["TUNE_ROWS"], len(Xtr)), replace=False)
    Xs, ys = Xtr[s], ytr[s]; d = Xtr.shape[1]; msq = max(1, int(round(math.sqrt(d))))
    def score(P, m, msl):
        f = WeightedRF(P, CFG["TUNE_K"], m, msl, 0).fit(Xs, ys)
        return f1_score(yva, f.predict_proba(Xva).argmax(1), average="macro")
    T = dict()
    # 1) RF tree hyper-parameters (shared by every forest variant -> fair comparison)
    grid = [(m, msl) for m in (msq, 2 * msq) for msl in (1, 3)]
    sc = {g: score(None, *g) for g in grid}; T["m"], T["msl"] = max(sc, key=sc.get)
    T["rf_grid"] = {f"m={k[0]},msl={k[1]}": v for k, v in sc.items()}
    # 2) NG-RF: tau at alpha=0.5, then alpha at best tau (coordinate search)
    sc = {t: score(to_prob(combine(imp["G"], imp["A"], 0.5), t), T["m"], T["msl"]) for t in CFG["TAU_GRID"]}
    T["tau"] = max(sc, key=sc.get)
    sc2 = {a: score(to_prob(combine(imp["G"], imp["A"], a), T["tau"]), T["m"], T["msl"]) for a in CFG["ALPHA_GRID"]}
    T["alpha"] = max(sc2, key=sc2.get); T["tau_val"] = sc; T["alpha_val"] = sc2
    # 3) every non-neural weighting gets its own tau (same budget as NG-RF)
    T["tau_w"] = {}
    for k, w in W.items():
        sc = {t: score(to_prob(w, t), T["m"], T["msl"]) for t in CFG["TAU_GRID"]}
        T["tau_w"][k] = max(sc, key=sc.get)
    # 4) CNN top-k hard selection: k chosen on validation
    I = combine(imp["G"], imp["A"], T["alpha"])
    sc = {}
    for frac in (0.25, 0.5, 0.75):
        k = max(2, int(round(frac * d))); top = np.argsort(-I)[:k]
        f = WeightedRF(None, CFG["TUNE_K"], min(T["m"], k), T["msl"], 0).fit(Xs[:, top], ys)
        sc[frac] = f1_score(yva, f.predict_proba(Xva[:, top]).argmax(1), average="macro")
    T["topk_frac"] = max(sc, key=sc.get)
    # 5) boosted trees
    sc = {}
    for dep in (6, 10):
        _, pr, _, _ = run_xgb(Xs, ys, Xva, yva, Xva, n_cls, dep, 0); sc[dep] = f1_score(yva, pr.argmax(1), average="macro")
    T["xgb_depth"] = max(sc, key=sc.get)
    sc = {}
    for lv in (63, 255):
        _, pr, _, _ = run_lgbm(Xs, ys, Xva, yva, Xva, n_cls, lv, 0); sc[lv] = f1_score(yva, pr.argmax(1), average="macro")
    T["lgbm_leaves"] = max(sc, key=sc.get)
    T = json.loads(json.dumps(T, default=float))
    json.dump(T, open(path, "w"), indent=1); log("tuning", ds, T)
    return T

In [ ]:
# %% ---------------------------------------------------------------- ONE REPEAT
def run_repeat(ds, rep, X, y, cats, task):
    """task: 'bin' (benign vs attack) or 'multi' (attack families)."""
    path = os.path.join(CKPT, f"{ds}_{task}_r{rep}.pkl")
    if os.path.exists(path): return pickle.load(open(path, "rb"))
    seed = CFG["SEED"] + rep; seed_all(seed)
    tr, va, te = make_split(X, y, rep)
    pp = Prep(cats).fit(X.iloc[tr])
    Xtr, Xva, Xte = pp.transform(X.iloc[tr]), pp.transform(X.iloc[va]), pp.transform(X.iloc[te])
    if task == "bin":
        classes = ["Benign", "Attack"]; yy = (y != "Benign").astype(int).values
    else:
        le = LabelEncoder().fit(y); classes = list(le.classes_); yy = le.transform(y)
    ytr, yva, yte = yy[tr], yy[va], yy[te]
    n_cls, binary = len(classes), task == "bin"
    R = dict(ds=ds, rep=rep, task=task, classes=classes, n_tr=len(tr), n_va=len(va), n_te=len(te),
             features=list(pp.cols), metrics={}, extra={})

    log(ds, task, "rep", rep, "CNN importances")
    imp = learn_importances(Xtr, ytr, Xva, yva, n_cls, seed)
    R["extra"]["G"], R["extra"]["A"], R["extra"]["perm_rho"] = imp["G"].tolist(), imp["A"].tolist(), imp["perm_rho"]
    R["extra"]["G_list"] = [g.tolist() for g in imp["G_list"]]; R["extra"]["A_list"] = [a.tolist() for a in imp["A_list"]]
    R["extra"]["cnn_params"] = n_params(imp["nets"][0][0]); R["extra"]["cnn_epochs"] = imp["nets"][0][2]
    R["extra"]["cnn_min"] = imp["cnn_secs"] / 60

    # uniform RF (needed for tuning weights) with provisional params
    d = Xtr.shape[1]; msq = max(1, int(round(math.sqrt(d))))
    tpath = os.path.join(CKPT, f"{ds}_tuning.json")
    if task == "bin" and not os.path.exists(tpath):
        rf0 = WeightedRF(None, CFG["TUNE_K"], msq, 1, seed).fit(Xtr[:min(len(Xtr), CFG["TUNE_ROWS"])], ytr[:min(len(Xtr), CFG["TUNE_ROWS"])])
        xgb0, _, _, _ = run_xgb(Xtr[:CFG["TUNE_ROWS"]], ytr[:CFG["TUNE_ROWS"]], Xva, yva, Xva, n_cls, 6, seed)
        W0 = importance_baselines(rf0, Xtr, ytr, Xva, yva, xgb0, seed)
        tune(ds, Xtr, ytr, Xva, yva, n_cls, imp, W0)
    T = json.load(open(tpath))
    K, m, msl = CFG["K"], T["m"], T["msl"]

    # ---- forests
    rf, R["metrics"]["RF"] = fit_eval_forest(None, Xtr, ytr, Xte, yte, K, m, msl, seed, binary=binary, classes=classes)
    R["extra"]["size_RF"] = rf.size_mb(); R["extra"]["div_RF"] = rf.diversity(Xva, seed=seed)
    R["extra"]["usage_RF"] = rf.feature_usage(d).tolist()
    _, R["metrics"]["ExtraTrees"] = fit_eval_forest(None, Xtr, ytr, Xte, yte, K, m, msl, seed, forest="et", binary=binary, classes=classes)

    I = combine(imp["G"], imp["A"], T["alpha"]); P = to_prob(I, T["tau"])
    ng, R["metrics"]["NG-RF"] = fit_eval_forest(P, Xtr, ytr, Xte, yte, K, m, msl, seed, binary=binary, classes=classes)
    R["metrics"]["NG-RF"]["train_min"] += imp["cnn_secs"] / 60           # include CNN cost
    R["extra"]["size_NG-RF"] = ng.size_mb(); R["extra"]["div_NG-RF"] = ng.diversity(Xva, seed=seed)
    R["extra"]["usage_NG-RF"] = ng.feature_usage(d).tolist(); R["extra"]["P"] = P.tolist()
    del ng; gc.collect()

    # ---- boosted trees
    xg, pr, ttr, tte = run_xgb(Xtr, ytr, Xva, yva, Xte, n_cls, T["xgb_depth"], seed)
    R["metrics"]["XGBoost"] = (binary_metrics(yte, pr[:, 1]) if binary else multiclass_metrics(yte, pr.argmax(1), classes))
    R["metrics"]["XGBoost"].update(train_min=ttr / 60, test_ms=tte)
    R["extra"]["size_XGBoost"] = len(xg.get_booster().save_raw()) / 2 ** 20
    lg, pr, ttr, tte = run_lgbm(Xtr, ytr, Xva, yva, Xte, n_cls, T["lgbm_leaves"], seed)
    R["metrics"]["LightGBM"] = (binary_metrics(yte, pr[:, 1]) if binary else multiclass_metrics(yte, pr.argmax(1), classes))
    R["metrics"]["LightGBM"].update(train_min=ttr / 60, test_ms=tte)
    R["extra"]["size_LightGBM"] = len(lg.booster_.model_to_string().encode()) / 2 ** 20; del lg

    # ---- closest work: forests weighted by non-neural importances
    W = importance_baselines(rf, Xtr, ytr, Xva, yva, xg, seed)
    for k, w in W.items():
        _, R["metrics"][k] = fit_eval_forest(to_prob(w, T["tau_w"][k]), Xtr, ytr, Xte, yte, K, m, msl, seed, binary=binary, classes=classes)
    R["extra"]["W"] = {k: v.tolist() for k, v in W.items()}
    kk = max(2, int(round(T["topk_frac"] * d))); top = np.argsort(-I)[:kk]
    _, R["metrics"]["CNN-FS+RF"] = fit_eval_forest(None, Xtr[:, top], ytr, Xte[:, top], yte, K, min(m, kk), msl, seed, binary=binary, classes=classes)
    del rf; gc.collect()

    # ---- neural baselines
    net, perm_, _ = imp["nets"][0]
    R["metrics"]["CNN"] = eval_net(net, Xte, yte, binary, classes); R["metrics"]["CNN"]["train_min"] = imp["cnn_secs"] / CFG["N_PERM"] / 60
    R["extra"]["size_CNN"] = n_params(net) * 4 / 2 ** 20
    if CFG["RUN_SEQ_BASELINES"] and (binary or CFG.get("SEQ_MULTI", True)):
        for name, Net in (("BiLSTM", BiLSTM), ("CNN-LSTM", CNNLSTM)):
            n2, t, _ = train_net(Net(d, n_cls), Xtr, ytr, Xva, yva, n_cls, seed)
            R["metrics"][name] = eval_net(n2, Xte, yte, binary, classes); R["metrics"][name]["train_min"] = t / 60
            R["extra"]["size_" + name] = n_params(n2) * 4 / 2 ** 20
            del n2
    del imp; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    pickle.dump(R, open(path, "wb")); log(ds, task, "rep", rep, "done", {k: round(v.get("f1", v.get("macro_f1", 0)), 3) for k, v in R["metrics"].items()})
    return R

In [ ]:
# %% ---------------------------------------------------------------- ABLATIONS (binary task)
def run_ablation(ds, rep, X, y, cats):
    path = os.path.join(CKPT, f"{ds}_abl_r{rep}.pkl")
    if os.path.exists(path): return pickle.load(open(path, "rb"))
    seed = CFG["SEED"] + rep; seed_all(seed)
    tr, va, te = make_split(X, y, rep)
    pp = Prep(cats).fit(X.iloc[tr])
    Xtr, Xva, Xte = pp.transform(X.iloc[tr]), pp.transform(X.iloc[va]), pp.transform(X.iloc[te])
    yy = (y != "Benign").astype(int).values; ytr, yva, yte = yy[tr], yy[va], yy[te]
    T = json.load(open(os.path.join(CKPT, f"{ds}_tuning.json"))); m, msl = T["m"], T["msl"]
    A = dict(alpha={}, tau={}, K={}, depth={}, perm={})
    bpath = os.path.join(CKPT, f"{ds}_bin_r{rep}.pkl")      # same split and seed -> reuse the relevance of the main run
    Rb = pickle.load(open(bpath, "rb")) if os.path.exists(bpath) else None
    if Rb is not None and "G_list" in Rb["extra"]:
        ex = Rb["extra"]
        imp = dict(G=np.array(ex["G"]), A=np.array(ex["A"]), G_list=[np.array(g) for g in ex["G_list"]],
                   A_list=[np.array(a) for a in ex["A_list"]], perm_rho=ex["perm_rho"])
    else:
        imp = learn_importances(Xtr, ytr, Xva, yva, 2, seed)
    for a in CFG["ALPHA_GRID"]:
        _, A["alpha"][a] = fit_eval_forest(to_prob(combine(imp["G"], imp["A"], a), T["tau"]), Xtr, ytr, Xte, yte, CFG["K"], m, msl, seed)
    for t in CFG["TAU_GRID"] + [float("inf")]:
        f, r = fit_eval_forest(to_prob(combine(imp["G"], imp["A"], T["alpha"]), t), Xtr, ytr, Xte, yte, CFG["K"], m, msl, seed)
        r["div"] = f.diversity(Xva, seed=seed); A["tau"][t] = r; del f
    P = to_prob(combine(imp["G"], imp["A"], T["alpha"]), T["tau"])
    for k in (25, 50, 100, 200):
        f, r = fit_eval_forest(P, Xtr, ytr, Xte, yte, k, m, msl, seed); r["size"] = f.size_mb(); A["K"][k] = r; del f
    # order invariance: single identity-order CNN vs N_PERM-averaged
    P1 = to_prob(combine(imp["G_list"][0], imp["A_list"][0], T["alpha"]), T["tau"])
    _, A["perm"][1] = fit_eval_forest(P1, Xtr, ytr, Xte, yte, CFG["K"], m, msl, seed)
    A["perm"][CFG["N_PERM"]] = A["alpha"][T["alpha"]] if T["alpha"] in A["alpha"] else None
    A["perm_rho"] = imp["perm_rho"]
    for name, ch in (("Shallow (32-64)", (32, 64)), ("Medium (64-128-256)", (64, 128, 256)), ("Deep (64-128-256-512)", (64, 128, 256, 512))):
        im = learn_importances(Xtr, ytr, Xva, yva, 2, seed, n_perm=1, chans=ch)
        _, r = fit_eval_forest(to_prob(combine(im["G"], im["A"], T["alpha"]), T["tau"]), Xtr, ytr, Xte, yte, CFG["K"], m, msl, seed)
        r["params"] = n_params(im["nets"][0][0]); r["cnn_min"] = im["cnn_secs"] / 60; A["depth"][name] = r
    pickle.dump(A, open(path, "wb")); log(ds, "ablation rep", rep, "done")
    return A

In [ ]:
# %% ---------------------------------------------------------------- STATISTICS
def corrected_t(a, b, n_tr, n_te):
    """Nadeau & Bengio (2003) corrected resampled t-test, two-sided."""
    d = np.asarray(a) - np.asarray(b); J = len(d)
    if J < 2 or np.all(d == d[0]): return float("nan"), 1.0 if np.all(d == 0) else 0.0
    v = np.var(d, ddof=1); t = d.mean() / math.sqrt((1 / J + n_te / n_tr) * v) if v > 0 else float("inf")
    p = 2 * stats.t.sf(abs(t), J - 1); return float(t), float(p)

def holm(pvals):
    keys = sorted(pvals, key=pvals.get); m = len(keys); out, run = {}, 0.0
    for i, k in enumerate(keys):
        run = max(run, min(1.0, (m - i) * pvals[k])); out[k] = run
    return out

In [ ]:
# %% ---------------------------------------------------------------- MAIN LOOP
if __name__ == "__main__":
    t_all = time.time()
    RAW = {}
    rawp = os.path.join(CFG["OUT_DIR"], "results_raw.pkl")
    if os.path.exists(rawp): RAW = pickle.load(open(rawp, "rb"))
    for ds in CFG["DATASETS"]:
        X, y, cats, info = prepare(ds)
        RAW.setdefault(ds, {})["info"] = info
        RAW[ds]["bin"] = [run_repeat(ds, r, X, y, cats, "bin") for r in range(CFG["REPEATS"])]
        RAW[ds]["multi"] = [run_repeat(ds, r, X, y, cats, "multi") for r in range(CFG["REPEATS"])]
        if CFG["RUN_ABLATIONS"]:
            RAW[ds]["abl"] = [run_ablation(ds, r, X, y, cats) for r in range(CFG["ABL_REPEATS"])]
        RAW[ds]["tuning"] = json.load(open(os.path.join(CKPT, f"{ds}_tuning.json")))
        RAW["env"] = dict(cpu=platform.processor() or platform.machine(), cores=os.cpu_count(),
                          gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
                          ram_gb=round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2 ** 30) if hasattr(os, "sysconf") else None,
                          python=platform.python_version(), torch=torch.__version__,
                          sklearn=__import__("sklearn").__version__, xgboost=__import__("xgboost").__version__,
                          lightgbm=__import__("lightgbm").__version__, cfg={k: v for k, v in CFG.items() if not k.endswith(("GLOB", "TRAIN", "TEST", "DIR"))})
        pickle.dump(RAW, open(rawp, "wb"))
        del X, y; gc.collect()
    log("all experiments finished in %.1f h" % ((time.time() - t_all) / 3600))
    # summarise -> tokens.json + figures (separate script so it can be re-run cheaply)
    import ngrf_report
    ngrf_report.main(CFG["OUT_DIR"])